# read

> Read PDFs, scans and page images page by page, with a vision model where the text layer cannot be trusted, and keep every reading of every page

In [ ]:
#| default_exp read

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

Many PDFs carry a text layer, which is the text you can select and copy. A PDF made from a digital document has an exact text layer, and reading it costs nothing. A scanned PDF has no text layer, or it has one made by OCR (optical character recognition), which can be poor. Old books in Indian scripts are often worse. Many were set in legacy fonts. The text layer of such a font reads as Latin gibberish, or it splits apart akṣaras, the letter clusters of Devanagari. A vision model reads images, so it can read such a page from its picture. It costs time or money for every page, and it makes mistakes of its own.

`read_doc` decides page by page. The `parse` argument sets the route:

| `parse=` | what each page gets |
|---|---|
| `auto`, the default | a text layer that passes the checks below is kept; a scanned page is OCRed; a page still scanned or flagged after that goes to `model` |
| `text` | the text layer only; a scanned page stays empty |
| `ocr` | OCR on every page |
| `llm` | `model` on every page |

`model` is any model id that the `urai` package accepts, local or hosted, such as `claude/sonnet` or `gemini/gemini-2.5-flash`. `None` means the value of `$VISHALAKSHI_MODEL`.

There are two entry points. The function `read_doc(path)` reads one file and returns its pages. It does not use a vault. `Vault.read_doc(src)` reads a file, or every PDF and image under a folder, into the vault. Each file becomes one document. The vault keeps the text, route and flags of each page as numbered versions. A new reading of a page never removes an earlier reading.

In [ ]:
#| export
import io, json, re, threading, time
from concurrent.futures import ThreadPoolExecutor
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from pdflite import pdf_doc, pdf_pages
from urai.record import resp_text
from litesearch import write_txn
from vishalakshi.core import Vault
from vishalakshi.pii import LEVEL_LABELS, pii_report, top, type_policy

## Prompts

A reading prompt tells a vision model what to return. `PRESETS` holds four reading prompts. Each asks for the page as JSON. The JSON is a list of items in reading order. An item is a heading, a paragraph of prose, a verse, a table, a note or a colophon, which is the line that closes a chapter in an Indian text. Each item has its text exactly as printed and any number printed with it. This page calls the items of a read page blocks.

| preset | use |
|---|---|
| `generic` | any printed page |
| `sanskrit` | Sanskrit prints. It names the letters that old prints make easy to misread. It tells the model to leave out translation and commentary |
| `tamil` | Tamil prints. It names the letters that old prints make easy to misread |
| `tables` | tables, read as rows of cells |

Every preset also asks for a `box` per item. The `box` is the region that the item covers, as `[x0, y0, x1, y1]`, in fractions of the page width and height from the top left. An app can use it to crop the block out of the page image or to highlight it.

`reading_prompt(name, title, script, hint)` fills a preset with the title of the document, its `script` (Devanagari by default) and a `hint`. A hint is a sentence about this book, such as how it numbers its verses or what to skip. A `name` that is not one of the four presets is the whole prompt. `reading_prompt` fills `{of}`, `{title}`, `{script}` and `{hint}` in it. If the prompt has JSON braces that are not doubled, formatting fails and `reading_prompt` returns the prompt unchanged.

In [ ]:
#| export
BOX = ('Give each item a "box": [x0, y0, x1, y1], the region it covers as fractions 0 to 1 of the page width and height from the top left; '
       'leave it out when unsure. ')
PRESETS = dict(
    generic=("You transcribe a scanned page{of}. Return JSON only, no prose, no code fence: "
             '{{"items": [{{"kind": "heading"|"prose"|"verse"|"table"|"note", "n": <a number printed with it, or null>, "text": "<exactly as printed>", '
             '"rows": [["cell", …], …], "box": [x0, y0, x1, y1]}}]}} in reading order. A heading is a title set on its own line. Prose: one item per paragraph, its lines joined '
             "with a space, a word hyphenated at a line end rejoined. A verse keeps its printed lines joined by \\n. A table gives `rows`, one list of cells "
             "per printed row, and an empty `text`. Skip running heads, page numbers and library stamps. Keep the spelling and script as printed and never "
             "rewrite a clear reading. " + BOX + "{hint} If the page has no text, return {{\"items\": []}}."),
    sanskrit=("You transcribe scanned pages of a printed {script} text{of}. Return JSON only, no prose, no code fence: "
              '{{"items": [{{"kind": "verse"|"prose"|"heading"|"colophon", "n": <the verse number printed at its end, or null>, "text": "<exactly as printed>", "box": [x0, y0, x1, y1]}}]}} '
              "in reading order. Transcribe ONLY the original-language text: verses, prose, a chapter heading in that language (e.g. अथ … अध्यायः), and a "
              "chapter-end colophon (इति …). Ignore English translation, commentary, notes, running heads and page numbers. For a verse, join its printed "
              "lines with \\n and keep the closing ॥ n ॥. A verse that runs off the bottom of the page: give its lines with n null. Read every akṣara "
              "carefully: this kind of typeface makes ष look like प, ी like ो, द्व like ह, and conjuncts like म्र, द्य, ष्ण, र्द्ध easy to drop. Where the print "
              "is ambiguous choose the reading that is a real word and scans, but never rewrite a clear reading. Keep sandhi, avagraha and daṇḍas as "
              "printed. " + BOX + "{hint} If the page has no such text, return {{\"items\": []}}."),
    tamil=("You transcribe scanned pages of a printed Tamil book{of}. Return JSON only, no prose, no code fence: "
           '{{"items": [{{"kind": "verse"|"prose"|"heading"|"colophon", "n": <a number printed with it, as an integer, or null>, "text": "<exactly as printed>", "box": [x0, y0, x1, y1]}}]}} '
           "in reading order. A verse keeps its printed lines joined by \\n; prose is one item per paragraph, its lines joined with a space. Write old "
           "letter forms in modern Unicode (the old ணா, றா, லை, னை glyphs as ணா, றா, லை, னை) and keep the spelling as printed. Old print confuses ண/ன, "
           "ள/ழ/ல, ற/ர and drops the puḷḷi: read each word as the Tamil it must be, but never rewrite a clear reading. Skip running heads and page "
           "numbers. " + BOX + "{hint} If the page has no such text, return {{\"items\": []}}."),
    tables=("You transcribe the tables on a scanned page{of}. Return JSON only, no prose, no code fence: "
            '{{"items": [{{"kind": "table", "title": <the heading over it or null>, "rows": [["cell", …], …], "box": [x0, y0, x1, y1]}} | {{"kind": "prose", "text": …, "box": …}}]}} '
            "in reading order, header row first, each cell exactly as printed. Text that is not a table is prose. " + BOX + "{hint} If the page has no text, "
            "return {{\"items\": []}}."))

def reading_prompt(name='generic', title='', script='', hint='') -> str:
    "Fill in the reading prompt `name` with the title, script and hint, or use `name` itself as the prompt when it is not a preset."
    s = PRESETS.get(name, name)
    try: return s.format(of=f': {title}' if title else '', title=title, script=script or 'Devanagari', hint=hint)
    except (KeyError, IndexError, ValueError): return s

In [ ]:
assert 'Gita' in reading_prompt('sanskrit', 'Gita') and 'ष look like प' in reading_prompt('sanskrit')
assert 'Tamil' in reading_prompt('tamil')
assert all('"box": [x0, y0, x1, y1]' in reading_prompt(k) and 'leave it out' in reading_prompt(k) for k in PRESETS)
test_eq(reading_prompt('Read it: {hint}', hint='x'), 'Read it: x')
test_eq(reading_prompt('A {script} page of {title}. {{"items": []}} {hint}', 'Gita', 'Tamil'), 'A Tamil page of Gita. {"items": []} ')
test_eq(reading_prompt('Return {"items": []} only.', 'Gita'), 'Return {"items": []} only.')   # no template: as it is

## Pages as images

A vision model receives a picture of the page. `render` draws one page of a PDF as a PNG at `dpi` (200 by default). Some hosted models refuse large images. When the PNG is larger than `max_bytes` (3 MB), `render` returns a JPEG with a long side of at most `max_px` (3,000 pixels). `fit` does the same for an image file.

Two cuts help with hard pages. `halves` splits a landscape image into a left page and a right page. A landscape image is usually a scan of two facing pages, called a spread. `page_strips` cuts a tall page into overlapping horizontal strips. The model then sees worn or small type at a larger scale.

In [ ]:
#| export
def render(pdf, page:int, dpi:int=200, max_px:int=3000, max_bytes:int=3_000_000) -> bytes:
    "Page `page` (0-based) as PNG, or as JPEG within `max_px` when the PNG is over `max_bytes`."
    d = pdf_doc(pdf)
    b = d.render_page(page, dpi=dpi, format='png')
    return b if len(b) <= max_bytes else d.render_page_fit(page, max_px, max_px, format='jpeg')

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.webp')

def _image(img):
    from PIL import Image
    return Image.open(io.BytesIO(img))

def _as_png(im): b = io.BytesIO(); im.save(b, 'PNG'); return b.getvalue()

def fit(img:bytes, max_px:int=3000, max_bytes:int=3_000_000) -> bytes:
    "Return `img` unchanged when it is within `max_bytes`, else as a JPEG whose long side fits `max_px`."
    if len(img) <= max_bytes: return img
    im = _image(img).convert('RGB'); im.thumbnail((max_px, max_px))
    b = io.BytesIO(); im.save(b, 'JPEG', quality=90); return b.getvalue()

def halves(img:bytes) -> list:
    "Split a landscape page image, taken to be a two-page spread, into its left and right halves, and return any other image as it is."
    im = _image(img); w, h = im.size
    return [_as_png(im.crop((0, 0, w//2, h))), _as_png(im.crop((w//2, 0, w, h)))] if w > h else [img]

def _wide(src, page:int) -> bool:
    "Whether `halves` cuts page `page` of `src`: an image by its own size, a PDF page by its box, turned by its rotation, without rendering it."
    p = Path(src)
    if p.suffix.lower() in IMG_EXTS: w, h = _image(p.read_bytes()).size; return w > h
    d = pdf_doc(p); x0, y0, x1, y1 = d.page_crop_box(page) or d.page_media_box(page)
    return (x1 - x0 > y1 - y0) != (d.page_rotation(page) % 180 == 90)

def page_strips(img:bytes, k:int, over:float=.06) -> list:
    "Cut `img` into `k` horizontal strips, top to bottom, each overlapping the next by `over` of the page height."
    im = _image(img); w, h = im.size; step = h / k
    return [_as_png(im.crop((0, max(0, int(i*step - over*h)), w, min(h, int((i+1)*step + over*h))))) for i in range(k)]

In [ ]:
from pdf_oxide import Pdf
_fx = Path(tempfile.mkdtemp())
for _i, _t in enumerate(['# Attention Is All You Need\n\nThe dominant sequence transduction models are based on recurrent or convolutional networks.',
                         '# Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.']):
    Pdf.from_markdown(_t).save(str(_fx/f'p{_i}.pdf'))
Pdf.merge([str(_fx/'p0.pdf'), str(_fx/'p1.pdf')]).save(str(_fx/'two.pdf'))
_src = _fx/'two.pdf'
_png = render(_src, 0, dpi=72)
assert _png[:4] == b'\x89PNG'
assert render(_src, 0, dpi=300, max_bytes=1000)[:2] == b'\xff\xd8'   # over the limit: a JPEG
test_eq(fit(_png), _png); assert fit(_png, 100, 1000)[:2] == b'\xff\xd8'
from PIL import Image
def _size(b): return Image.open(io.BytesIO(b)).size
_w, _h = _size(_png)
test_eq(halves(_png), [_png])                                   # a portrait page is one page
_b = io.BytesIO(); Image.new('L', (200, 100), 255).save(_b, 'PNG'); _spread = _b.getvalue()
test_eq([_size(x) for x in halves(_spread)], [(100, 100), (100, 100)])
test_eq([_size(x) for x in page_strips(_png, 3)], [(_w, int(_h/3 + .06*_h)), (_w, int(2*_h/3 + .06*_h) - int(_h/3 - .06*_h)), (_w, _h - int(2*_h/3 - .06*_h))])

## Checks

`flags` lists what looks wrong with a page's text. It is how `parse='auto'` decides that a text layer cannot be trusted.

| flag | meaning |
|---|---|
| `thin` | fewer than `min_chars` (20) letters, as on a scanned page |
| `replacement` | the Unicode replacement character, which marks text that failed to decode |
| `legacy` | more than 5 Latin-1 or typographic characters per 100 letters, which is how a legacy Devanagari font's text layer reads |
| `split` | a line that starts with a vowel sign or virāma, a piece of an akṣara cut off from the rest |
| `akshara` | an akṣara that cannot be written, such as a virāma followed by a vowel sign |
| `foreign` | Greek or Cyrillic letters next to Devanagari |

`disagree(a, b)` measures how far two readings of the same text differ: the letter edits between them divided by the letters in the longer one. Spaces, digits, daṇḍas and punctuation are ignored and vowel signs count. A value of 0.1 or more is worth a look.

In [ ]:
#| export
DEP = re.compile('(?m)^[ऀ-ःऺ-़ा-्ॎॏ॑-ॗॢॣா-்ௗ]')                    # a line opening on a dependent sign: a split akṣara
AKSHARA = re.compile('्[ा-ौॢॣ]|[ा-ॄॆ-ौ][ा-ॄॆ-ौ]|(?<!ग)्[ंः]|[ा-्]़|[ऄ-औ][ा-ौ]')   # malformed Devanagari akṣaras
FOREIGN = re.compile('[ऀ-ॿ][Ͱ-ϿЀ-ӿ]|[Ͱ-ϿЀ-ӿ][ऀ-ॿ]')
LEGACY = re.compile('[¡-ÿ‘-„†-‰]')       # what a legacy Devanagari font's text layer reads as
def flags(text:str, min_chars:int=20) -> list:
    "List what looks wrong with a page's text: `thin`, `replacement`, `legacy`, `split`, `akshara` or `foreign`."
    t = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', text or '')
    letters = re.sub(r'[\W\d_]', '', t)
    out = []
    if len(letters) < min_chars: out.append('thin')
    if '�' in t: out.append('replacement')
    if letters and len(LEGACY.findall(t)) > .05 * len(letters): out.append('legacy')
    if DEP.search(t): out.append('split')
    if AKSHARA.search(t): out.append('akshara')
    if FOREIGN.search(t): out.append('foreign')
    return out

_LETTERS = re.compile(r'[\s\d०-९।॥|ऽ\u200c\u200d!-/:-@\[-`{-~–—‘’“”]')   # spaces, digits, daṇḍas, avagraha, joiners, punctuation; vowel signs stay
def disagree(a:str, b:str) -> float|None:
    "How far two readings differ: letter edits divided by the longer reading's letters, or None when neither has any."
    a, b = _LETTERS.sub('', a or ''), _LETTERS.sub('', b or '')
    if not (a or b): return None
    p = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        c = [i]
        for j, y in enumerate(b, 1): c.append(min(p[j] + 1, c[j-1] + 1, p[j-1] + (x != y)))
        p = c
    return p[-1] / max(len(a), len(b))

In [ ]:
test_eq(flags('Hello world this is enough letters'), [])
test_eq(flags('क्ा and more letters here to pass'), ['akshara'])
assert 'split' in flags('कृष्ण\nा is a sign broken off here, long enough')
assert 'legacy' in flags('Jke ‘kfDr ¼ji¼ Hkxoku ¾ ™ more text here')
test_eq(flags(''), ['thin'])
assert disagree('रामः', 'रामा') > 0 and disagree('राम ॥१॥', 'राम') == 0
test_eq(disagree('', ''), None)

## Reading a page

`read_page` sends one page image to a vision model. It returns `blocks`, their text joined as markdown (`text`), the `model` and the `cost` that the backend reported.

With no `chat=`, the call goes through `CachedChat`. `CachedChat` records each reading once and replays it later. A repeated read of the same page costs nothing. The recordings are in `chatcache` in the working directory, or in the folder that `$VISHALAKSHI_CHAT_CACHE` names. A page with no recording reaches the model only when `$VISHALAKSHI_RECORD_CHAT` is set. Otherwise `read_page` raises an error that says so. `fresh=True` drops the recorded reply and asks the model again. A reply that is not JSON raises an error and is not recorded, and the next try reaches the model. `chat=` takes any callable in place of a model. The tests use it to run without a model. `fit` limits the size of every image before it is sent.

A block keeps its `box` only when the box is four numbers that make a real rectangle. `read_page` clamps the box to the page. It drops any other box. `crop` cuts a block out of the page image by its box.

Before it sends an image, `read_page` asks `send_rule` whether the page may go. The level of the page is the level of its document. It rises to Personal when `text` holds an identifier. `text` is the text layer or the OCR text of the page. A page image cannot be masked. A Personal page goes to a hosted model only with `allow=True` or `pii='off'`. A Local only page never goes to a hosted model. A page that may not be sent comes back empty, and `blocked` says why.

In [ ]:
#| export
def _md(items) -> str:
    "A page's items as markdown."
    def one(it):
        if it.get('rows'): return '\n'.join('| ' + ' | '.join(map(str, r)) + ' |' + ('\n|' + '---|' * len(r) if not i else '') for i, r in enumerate(it['rows']))
        t = (it.get('text') or '').strip()
        return f'## {t}' if it.get('kind') == 'heading' and t else t
    return '\n\n'.join(s for s in map(one, items or []) if s)

def _json(s):
    s = re.sub(r'^```(?:json)?\s*|\s*```$', '', (s or '').strip())
    return json.loads(s[s.find('{'):s.rfind('}') + 1])

def sane_box(b) -> list|None:
    "`b` as `[x0, y0, x1, y1]` clamped to 0-1 with `x0<x1` and `y0<y1`, or None."
    try: x0, y0, x1, y1 = (min(1., max(0., float(v))) for v in b)
    except (TypeError, ValueError): return None
    return [x0, y0, x1, y1] if x0 < x1 and y0 < y1 else None

def _boxed(items):
    for it in items:
        if isinstance(it, dict) and 'box' in it and (b := sane_box(it.pop('box'))): it['box'] = b
    return items

def crop(img:bytes, box, pad:float=.02) -> bytes:
    "Cut the region `box` out of a page image, with a margin of `pad` of the page, as PNG; return the whole image when the box is unusable or PIL is missing."
    try: from PIL import Image
    except ImportError: return img
    if not (b := sane_box(box)): return img
    im = Image.open(io.BytesIO(img)); w, h = im.size
    l, t = int(max(0, b[0] - pad) * w), int(max(0, b[1] - pad) * h)
    r, d = max(l + 1, round(min(1, b[2] + pad) * w)), max(t + 1, round(min(1, b[3] + pad) * h))
    out = io.BytesIO(); im.crop((l, t, r, d)).save(out, 'PNG')
    return out.getvalue()

def backends():
    "Register the `rishi` model backends with `urai`, so that ids such as `claude/sonnet` resolve."
    import rishi

def _forget(c, msg):
    "Drop a recorder's reply to `msg`, so the next ask reaches the model."
    if hasattr(c, 'rec') and hasattr(c, '_key'): c.rec.forget(c._key('call', msg, {}))

def read_page(img:bytes, model:str=None, preset:str='generic', title:str='', script:str='', hint:str='', chat=None,
              pii:str='redact',   # off|local|redact|refuse; an image cannot be masked, so a Personal page reaches a hosted model only with `allow` or `off`
              level:str='open',   # the page's level, from its document; raised to Personal when `text` holds identifiers
              text:str='',        # the page's text layer or OCR, to judge it by
              allow:bool=False,   # allow once: a Personal page may go to a hosted model; a Local only one never does
              strips:int=1,       # sent as this many overlapping strips (`page_strips`), the model told to read the page once
              fresh:bool=False,   # ask the model again, not its recorded reply
) -> AttrDict:
    "Read one page image with a vision model and return its `blocks`, their markdown `text`, the `model` and the `cost`, or `blocked` when the page may not be sent."
    from vishalakshi.ask import CachedChat, model_local, send_rule
    # the title is part of the reading prompt, and is judged with the page
    lv = top(level, 'personal' if (s := f"{title or ''}\n{text or ''}".strip()) and pii_report(s, ner=True).has_pii else 'open')
    if send_rule(lv, model_local(chat if chat is not None else model), pii, allow, image=True) == 'withheld':
        return AttrDict(blocks=[], text='', model=getattr(chat, 'model', model), cost=None, blocked=AttrDict(
            kind='local_only' if lv == 'local' else 'pii_blocked', level=lv, why=f'a {LEVEL_LABELS[lv]} page is not sent to a hosted model'))
    if chat is None: backends()
    c = chat or CachedChat(model, sp=reading_prompt(preset, title, script, hint), max_output_tokens=16000)
    ask = 'Transcribe the text on this page.' + (f' The page comes as {strips} overlapping strips, top to bottom: transcribe the page once, giving '
                                                  'the lines the strips share only once.' if strips > 1 else '')
    msg = [*map(fit, page_strips(img, strips) if strips > 1 else [img]), ask]
    if fresh: _forget(c, msg)
    try:
        r = c(msg)
        try: items = _boxed(_json(resp_text(r) if not isinstance(r, str) else r).get('items') or [])
        except ValueError: _forget(c, msg); raise   # a reply that is not JSON is not kept to be replayed
    finally:
        if chat is None and hasattr(c, 'close'): c.close()
    return AttrDict(blocks=items, text=_md(items), model=getattr(c, 'model', model), cost=((r.get('usage') or {}).get('cost') if isinstance(r, dict) else None))

In [ ]:
class Fake:
    "A vision model stand-in: it answers every page with the same two items, in a code fence."
    model, calls = 'fake/vision', 0
    def __call__(self, msg):
        Fake.calls += 1
        return dict(content='```json\n{"items":[{"kind":"heading","text":"Attention Is All You Need"},{"kind":"prose","text":"The dominant sequence transduction models are based on recurrent networks."}]}\n```', usage=dict(cost=0.01))
_r = read_page(_png, chat=Fake())
test_eq([b['kind'] for b in _r.blocks], ['heading', 'prose'])
assert _r.text.startswith('## Attention') and _r.cost == .01
class Boxed(Fake):
    "A reader that gives boxes: one sane, one out of range, one reversed, one not a box."
    def __call__(self, msg):
        return dict(content=json.dumps(dict(items=[dict(kind='heading', text='Attention', box=[.1, .05, .9, .1]), dict(kind='prose', text='The dominant models', box=[-.1, .2, 1.2, .5]),
                                                   dict(kind='prose', text='are recurrent', box=[.9, .6, .1, .7]), dict(kind='note', text='x', box='top')])))
_r = read_page(_png, chat=Boxed())
test_eq([b.get('box') for b in _r.blocks], [[.1, .05, .9, .1], [0., .2, 1., .5], None, None])
from PIL import Image
_w, _h = Image.open(io.BytesIO(_png)).size
_c = crop(_png, [0, 0, .5, .5], pad=0)
test_eq((_c[:4], Image.open(io.BytesIO(_c)).size), (b'\x89PNG', (round(_w / 2), round(_h / 2))))
_cw, _ch = Image.open(io.BytesIO(crop(_png, [.4, .4, .6, .6]))).size
assert abs(_cw - .24 * _w) <= 1 and abs(_ch - .24 * _h) <= 1
test_eq(crop(_png, None), _png); test_eq(crop(_png, [.5, .5, .5, .6]), _png)
test_eq(_md([dict(kind='table', rows=[['a', 'b'], ['1', '2']])]), '| a | b |\n|---|---|\n| 1 | 2 |')

In [ ]:
#| hide
# without PIL a crop is the whole page
import sys
_pil, sys.modules['PIL'] = sys.modules.get('PIL'), None
try: test_eq(crop(_png, [0, 0, .5, .5]), _png)
finally: sys.modules['PIL'] = _pil

In [ ]:
backends()
from urai.opts import resolve_runtime
test_eq(resolve_runtime('claude/sonnet'), ('claude', 'sonnet'))

## Reading a document

`read_doc(path)` takes a PDF or a page image and returns one row per page: `{page, text, route, flags}`. A page that the model read also has `blocks`, `model` and `cost`. `route` says where the text came from. Its values are `text` (the text layer), `ocr`, `model`, and `scanned` when nothing read the page. `pages=[...]` limits which pages may go to the model. The other pages keep their text layer. `workers` (4 by default) sets how many pages are read at the same time.

Privacy works page by page, as in `read_page`. When any page is going to a model and `pii` is not `off`, `read_doc` first checks the whole text layer for a sensitive doctype. A document whose text clearly reads as a passport or a payslip is Local only before any page image leaves. `local_model=` or a built `local_chat=` reads the pages that the hosted `model` may not read. Without either, those pages keep their text layer and carry `blocked`. A scanned page has no text to check. Only the level of its document applies to it.

These options control a long read:

- `progress(done, total, page, route, cost)` is called once with `done=0`, then after each page that was due for the model. If it returns False, the reading stops. The pages not yet read keep their text layer or OCR. The result's `cancelled` is True, and `unread` lists those pages. `read_doc` ignores an error raised inside `progress`.
- `cache=` keeps each model reading as a JSON file, `{page, items, cost, model}`. The file is in a folder (`p0001.json`, or `p0001a.json` and `p0001b.json` for the two halves of a spread), or at the path that a function `cache(page, half)` returns. A cached page is not sent again. Its `cost` is None, because nothing was spent, and `cached` is True.
- `redo=[...]` reads those pages again even when they are cached. After the new reading is saved, the old reading moves to a sibling folder. The folder has the name of the cache folder with `_prev` added. A number is added to the file name when an older copy is already there.
- `spread=True` treats a landscape page as two facing pages and reads the halves in turn. `strips=k` sends each image as `k` overlapping strips.
- `clean=` is a function that `read_doc` applies to the text of each block before it keeps the block.
- `tries=n` tries a failing page up to `n` times. It waits `RETRY_WAIT` seconds (10) after the first failure, 20 after the second, and so on. A page that still fails keeps its text layer, carries its `error`, and is listed in the result's `failed`. Without `tries`, `read_doc` raises the first error.

`doc_cost(rows)` adds up the cost of the pages that a model read. `doc_files(folder)` lists the PDFs and images under a folder.

In [ ]:
#| export
PARSE = ('auto', 'text', 'ocr', 'llm')
RETRY_WAIT = 10   # seconds before a failed page's second try; twice that before its third
def as_list(x, f=str):
    "Turn `x` into a list of `f`, splitting a comma-separated string as a command line gives it; None stays None."
    return None if x is None else [f(v) for v in (x.split(',') if isinstance(x, str) else x) if str(v).strip()]
def read_doc(path, parse:str='auto', model:str=None, preset:str='generic', title:str='', script:str='', hint:str='',
             pages=None, dpi:int=200, out=None, workers:int=4, chat=None,
             pii:str='redact',      # what `read_page` applies to each page; `off` sends Personal pages too, never Local only ones
             level:str=None,        # the document's level; None -> open, raised to Local only when its text reads as a sensitive type
             allow:bool=False,      # allow once, for Personal pages
             local_model:str=None,  # reads the pages a hosted `model` may not; None -> those pages keep their text layer
             local_chat=None,       # a built local reader instead of `local_model`
             progress=None,         # `progress(done, total, page, route, cost)`; False stops the reading
             spread:bool=False,     # a landscape page is a two-page spread: its halves read in turn (`halves`)
             strips:int=1,          # each image sent as overlapping strips (`read_page`)
             cache=None,            # a folder, or `cache(page, half)` -> a path: each reading kept as JSON there and not sent again
             redo=None,             # pages read again though cached; the old reading moved to `<folder>_prev/` once the new one is in
             clean=None,            # `clean(text)` -> text, made of every block's text before it is kept
             tries:int=None,        # a page that fails this many times is left out and listed in `failed`; None raises its error
) -> L:
    "Read each page of a PDF or page image, keeping a sound text layer and sending scanned or flagged pages to `model`, and return one row per page."
    assert parse in PARSE, f'parse is one of {PARSE}'
    pages = as_list(pages, int)
    p = Path(path)
    if p.suffix.lower() in IMG_EXTS: rows = [dict(page=0, text='', route='scanned')]
    else: rows = pdf_pages(p, out or p.parent/'.read', ocr=dict(auto='auto', text='off', ocr='on', llm='off')[parse])
    rows = L(AttrDict(r, flags=flags(r['text'])) for r in rows)
    want, redo = (pages if pages is None else set(pages)), set(as_list(redo, int) or ())
    def needs(r): return parse == 'llm' or (parse == 'auto' and (r.route == 'scanned' or bool(r.flags)))
    todo = [r for r in rows if (want is None or r.page in want) and needs(r)]
    if todo and pii != 'off': level = top(level, type_policy(text='\n\n'.join(r.text for r in rows)[:6000]).level)
    skip, fail, done, lock = [], [], [0], threading.Lock()
    at = cache if cache is None or callable(cache) else (lambda page, half, d=Path(cache): d/f'p{page + 1:04d}{half}.json')
    def tell(r=None):
        if not progress: return
        with lock:
            done[0] += r is not None
            try: ok = progress(done[0], len(todo), r and r.page, r and r.route, r and r.get('cost'))
            except Exception: ok = None
        if ok is False: skip.append(None)
    def go(r):
        if skip: skip.append(r.page); return r
        one(r); tell(r)
        return r
    def kept(r):
        "The page's cached readings, its whole page's or its halves', in order; None when it has none or is to be read again."
        if not at or r.page in redo: return None
        if spread and _wide(p, r.page):   # a spread is kept only when both halves are: one alone is read again whole
            fs = [Path(at(r.page, h)) for h in ('a', 'b')]
            return [json.loads(f.read_text(encoding='utf-8')) for f in fs] if all(f.is_file() for f in fs) else None
        fs = [f for f in dict.fromkeys(Path(at(r.page, h)) for h in ('', 'a', 'b')) if f.is_file()]
        return [json.loads(f.read_text(encoding='utf-8')) for f in fs] or None
    def read(img, kw, fresh):
        got = read_page(img, model, preset, title, script, hint, chat=chat, strips=strips, fresh=fresh, **kw)
        if got.get('blocked') and (local_model or local_chat): got = read_page(img, local_model, preset, title, script, hint, chat=local_chat, strips=strips, fresh=fresh, **kw)
        if clean and not got.get('blocked'):
            for b in got.blocks: b['text'] = clean(b.get('text') or '')
        return got
    def keep(f, got):
        f = Path(f); f.parent.mkdir(parents=True, exist_ok=True)
        if f.is_file():
            d = f.parent.parent/f'{f.parent.name}_prev'; d.mkdir(exist_ok=True)
            to, k = d/f.name, 1
            while to.exists(): to, k = d/f'{f.stem}.{k}.json', k + 1
            f.rename(to)
        f.write_text(json.dumps(dict(page=f.stem, items=got.blocks, cost=got.cost, model=got.model), ensure_ascii=False, indent=1), encoding='utf-8')
    def one(r):
        if (old := kept(r)) is not None:
            bs = [b for o in old for b in o.get('items', o.get('blocks')) or []]
            if (t := _md(bs)).strip(): r.update(text=t, route='model', model=old[0].get('model'), cost=None, flags=flags(t), blocks=bs, cached=True)
            return r
        whole = spread or strips > 1   # cut from the page at full size, each piece fitted after
        img = p.read_bytes() if p.suffix.lower() in IMG_EXTS else render(p, r.page, dpi, max_bytes=float('inf') if whole else 3_000_000)
        ims = halves(img) if spread else [img if whole else fit(img)]
        kw, gots = dict(pii=pii, level=level or 'open', text=r.text, allow=allow), []
        for h, im in zip(('a', 'b') if len(ims) > 1 else ('',), ims):
            for k in range(tries or 1):
                try: got = read(im, kw, fresh=k > 0 or r.page in redo); break
                except Exception as e:
                    if tries is None: raise
                    r.error = f'{type(e).__name__}: {e}'; time.sleep(RETRY_WAIT * (k + 1))
            else:
                with lock: fail.append(r.page)
                return r
            if got.get('blocked'): r.blocked = got.blocked; return r
            gots.append((h, got))
        if at:
            for h, got in gots: keep(at(r.page, h), got)
        bs, cs = [b for _, g in gots for b in g.blocks], [g.cost for _, g in gots if g.cost is not None]
        if (t := _md(bs)).strip(): r.update(text=t, route='model', model=gots[0][1].model, cost=sum(cs) if cs else None, flags=flags(t), blocks=bs)
        r.pop('error', None)
        return r
    tell()
    if todo:
        with ThreadPoolExecutor(max(1, workers)) as ex: list(ex.map(go, todo))
    rows.unread = sorted(x for x in skip if x is not None)
    rows.cancelled = bool(rows.unread)
    rows.failed = sorted(fail)
    return rows

def doc_cost(rows) -> float:
    "The summed cost of the pages a model read."
    return sum(r.get('cost') or 0 for r in rows)

def doc_files(folder, exts=('.pdf',) + IMG_EXTS) -> L:
    "The PDFs and page images under `folder`, sorted."
    return L(sorted(f for f in Path(folder).rglob('*') if f.is_file() and f.suffix.lower() in exts and not any(s.startswith('.') for s in f.relative_to(folder).parts)))

In [ ]:
_d = Path(tempfile.mkdtemp())
_scan = _d/'scan.pdf'; _scan.write_bytes(pdf_doc(pdf_doc(_src).extract_pages_to_bytes([0])).flatten_to_images(dpi=150))
Fake.calls = 0
_t = read_doc(_src, chat=Fake(), out=_d/'.out')
test_eq(_t.attrgot('route'), ['text', 'text']); test_eq(Fake.calls, 0)          # a sound text layer goes to no model
_s = read_doc(_scan, parse='text', out=_d/'.out'); test_eq(_s[0].route, 'scanned')
_m = read_doc(_scan, parse='llm', chat=Fake(), out=_d/'.out')
test_eq(_m[0].route, 'model'); test_eq(_m[0].model, 'fake/vision')
(_d/'pg.png').write_bytes(_png)
test_eq(read_doc(_d/'pg.png', chat=Fake())[0].route, 'model')
test_eq(doc_files(_d).map(lambda f: f.name), ['pg.png', 'scan.pdf'])
_h = Path(tempfile.mkdtemp())/'.cache'/'in'; (_h/'.skip').mkdir(parents=True); (_h/'pg.png').write_bytes(_png); (_h/'.skip'/'x.png').write_bytes(_png)
test_eq(doc_files(_h).map(lambda f: f.name), ['pg.png'])   # only a hidden folder below the one asked for hides a file

In [ ]:
# progress after each model page; False stops the rest, which keep their text layer
_seen = []
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, progress=lambda *a: _seen.append(a))
test_eq(_seen, [(0, 2, None, None, None), (1, 2, 0, 'model', .01), (2, 2, 1, 'model', .01)])
test_eq((doc_cost(_m), _m.cancelled, _m.unread), (.02, False, []))
_seen = []
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, progress=lambda *a: _seen.append(a) or a[0] < 1)
test_eq((_m.attrgot('route'), _m.cancelled, _m.unread, doc_cost(_m), len(_seen)), (['model', 'text'], True, [1], .01, 2))
assert 'Extended Neural GPU' in _m[1].text
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', progress=lambda *a: False)
test_eq((_m.attrgot('route'), _m.unread, doc_cost(_m)), (['text', 'text'], [0, 1], 0))
def _bad(*a): raise ValueError
test_eq(read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', progress=_bad).attrgot('route'), ['model', 'model'])   # its errors are swallowed
test_eq(read_doc(_src, chat=Fake(), out=_d/'.out').cancelled, False)

In [ ]:
# a cache: each reading kept as JSON and never sent again; `redo` reads again and keeps the old aside; `clean` tidies each block first
_cd = Path(tempfile.mkdtemp())/'ocr'
Fake.calls = 0
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, cache=_cd, clean=str.upper)
test_eq((Fake.calls, sorted(f.name for f in _cd.iterdir()), _m.failed), (2, ['p0001.json', 'p0002.json'], []))
_j = json.loads((_cd/'p0001.json').read_text())
test_eq((_j['page'], _j['items'][0]['text'], _j['cost'], _j['model']), ('p0001', 'ATTENTION IS ALL YOU NEED', .01, 'fake/vision'))
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, cache=_cd)
test_eq((Fake.calls, _m.attrgot('route'), [r.get('cached') for r in _m], doc_cost(_m)), (2, ['model', 'model'], [True, True], 0))   # nothing sent, nothing spent
assert _m[0].text.startswith('## ATTENTION')
_m = read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, cache=_cd, redo=[1])
test_eq((Fake.calls, sorted(f.name for f in (_cd.parent/'ocr_prev').iterdir())), (3, ['p0002.json']))
assert _m[1].text.startswith('## Attention') and _m[0].text.startswith('## ATTENTION')
read_doc(_src, parse='llm', chat=Fake(), out=_d/'.out', workers=1, cache=_cd, redo='1')
test_eq(sorted(f.name for f in (_cd.parent/'ocr_prev').iterdir()), ['p0002.1.json', 'p0002.json'])
class Blank(Fake):
    def __call__(self, msg): Fake.calls += 1; return dict(content='{"items": []}')
Fake.calls, _ed = 0, Path(tempfile.mkdtemp())
read_doc(_src, parse='llm', chat=Blank(), out=_d/'.out', workers=1, cache=_ed); read_doc(_src, parse='llm', chat=Blank(), out=_d/'.out', workers=1, cache=_ed)
test_eq((Fake.calls, json.loads((_ed/'p0001.json').read_text())['items']), (2, []))   # an empty page is kept too, and not paid for again

In [ ]:
#| hide
# through the recorder: a reply that is not JSON is forgotten, and `fresh` asks again rather than replaying
from vishalakshi.ask import CachedChat
class Live:
    "The model behind a recorder: says nonsense first, then a page."
    n = 0
    def __init__(self, *a, **kw): pass
    def __call__(self, msg, **kw):
        Live.n += 1
        return dict(content='not json' if Live.n == 1 else json.dumps(dict(items=[dict(kind='prose', text=f'reading {Live.n}')])), usage=dict(cost=.1))
    hist, cancelled = [], False
    def _recreate_conv(self): pass
_rc = CachedChat('litert/vision', path=tempfile.mkdtemp(), record=True); _rc._chat = Live()
test_fail(lambda: read_page(_png, chat=_rc), contains='Expecting value')
test_eq(read_page(_png, chat=_rc).text, 'reading 2')       # the nonsense was not kept: asked again
test_eq(read_page(_png, chat=_rc).text, 'reading 2')       # a good reply is replayed
test_eq(read_page(_png, chat=_rc, fresh=True).text, 'reading 3')

In [ ]:
from PIL import Image

In [ ]:
# a spread read as its two pages, kept where `cache(page, half)` says; the page sent as strips; a failing page tried again, then left out
_spf = Path(tempfile.mkdtemp())/'spread.png'; Image.new('L', (400, 200), 255).save(_spf)
class Halves(Fake):
    "Reads each image it is sent as the next half."
    def __call__(self, msg):
        Fake.calls += 1; Halves.msg = msg
        return dict(content=json.dumps(dict(items=[dict(kind='verse', text=f'half {Fake.calls}')])), usage=dict(cost=.5))
Fake.calls, _hd = 0, Path(tempfile.mkdtemp())
_at = lambda page, half: _hd/f'p{page + 1:04d}{half or "a"}.json'
_m = read_doc(_spf, chat=Halves(), spread=True, cache=_at)
test_eq((_m[0].text, _m[0].cost, sorted(f.name for f in _hd.iterdir())), ('half 1\n\nhalf 2', 1., ['p0001a.json', 'p0001b.json']))
test_eq((read_doc(_spf, chat=Halves(), spread=True, cache=_at)[0].text, Fake.calls), ('half 1\n\nhalf 2', 2))
(_hd/'p0001b.json').unlink()   # one half kept is not the page: both read again, the old half set aside
test_eq((read_doc(_spf, chat=Halves(), spread=True, cache=_at)[0].text, Fake.calls, (_hd.parent/f'{_hd.name}_prev'/'p0001a.json').exists()), ('half 3\n\nhalf 4', 4, True))
_ppf = Path(tempfile.mkdtemp())/'tall.png'; Image.new('L', (200, 400), 255).save(_ppf)   # a portrait page of a spread document is one reading, kept whole
read_doc(_ppf, chat=Halves(), spread=True, cache=(_at2 := lambda page, half, d=Path(tempfile.mkdtemp()): d/f'p{page + 1:04d}{half or "a"}.json'))
test_eq((read_doc(_ppf, chat=Halves(), spread=True, cache=_at2)[0].text, Fake.calls), ('half 5', 5))
read_doc(_d/'pg.png', chat=Halves(), strips=3)
test_eq((len(Halves.msg), 'overlapping strips' in Halves.msg[-1]), (4, True))
class Flaky(Fake):
    "Fails its first call, as a model starting slowly does."
    n = 0
    def __call__(self, msg):
        Flaky.n += 1
        if Flaky.n == 1: raise TimeoutError('slow start')
        return super().__call__(msg)
class Down(Fake):
    def __call__(self, msg): raise TimeoutError('down')
RETRY_WAIT = 0
_m = read_doc(_src, parse='llm', chat=Flaky(), out=_d/'.out', workers=1, tries=2)
test_eq((_m.failed, _m.attrgot('route'), [r.get('error') for r in _m]), ([], ['model', 'model'], [None, None]))
_m = read_doc(_src, parse='llm', chat=Down(), out=_d/'.out', workers=1, tries=2, pages=[0], cache=(_fd := Path(tempfile.mkdtemp())))
test_eq((_m.failed, _m[0].route, _m[0].error, (_fd/'p0001.json').exists()), ([0], 'text', 'TimeoutError: down', False))
test_fail(lambda: read_doc(_src, parse='llm', chat=Down(), out=_d/'.out', workers=1), contains='down')
RETRY_WAIT = 10

## Into the vault

`Vault.read_doc(src)` reads a file, or every PDF and image under a folder. It adds each file to the vault as one document. Search and `ask` reach the document like any other. It also keeps each page as a version in the `read_pages` table. A page gets a new version only when its text, route or blocks changed. Reading a file twice with the same result adds nothing.

Each new version is scanned for personal information, and the findings are filed against it (see [pii](09_pii.ipynb)). A new document starts at the shelf's floor. A document already in the vault keeps its own level.

`Vault.pages(doc_id)` returns the latest version of each current page. `history=True` returns every version of those pages. A shorter file marks pages beyond its end stale. Their readings remain in `read_pages`. They are not indexed or returned by default. The vault writes every page that a hosted model read to the send log.

Use `stale=True` on `pages`, `search`, `sections`, `context`, `document`, `doc_context`, `ask` or `federate` to include retained stale pages. Retrieval builds a temporary in-memory index for this request. It never adds stale text to the stored index. Hybrid search can still return unrelated current pages for a missing word.

`prune_stale(doc_id=None, dry_run=False)` deletes every reading of stale pages on this shelf. It returns each document id, page number and version count. `dry_run=True` returns the same report without deleting anything. The method is Python-only, like `housekeep`.

The result of `Vault.read_doc` is the ingest result with these keys added: `routes` (how many pages took each route), `flagged`, `blocked`, `cost`, `model_pages`, `cancelled`, `unread` and `failed`.

`Vault.reparse(doc_id, pages=[...])` reads pages of a document again. By default it uses the model (`parse='llm'`) and the preset, script and hint of the document. It files the new readings as new versions and re-indexes the document. Use it when a page came out badly and a better model should read it. `reparse` returns the ingest result with these keys added: `pages` (the pages it read), `changed` (the pages whose text changed), `blocked`, `cost`, `model_pages`, `cancelled` and `unread`.

Both methods take `progress`. A cancelled or failed read of a document already in the vault leaves its unread and failed pages as they were. When a folder is read with `cache=`, each file caches in its own subfolder. The subfolder has the file's path below the folder, extension included, so `a/report.pdf` and `b/report.pdf` do not share one. A cache written by an earlier version, which named each file by its stem alone, is not reused. Those pages are read again.

In [ ]:
#| export
@patch
def _pages_t(self:Vault):
    t = self.db.t.read_pages
    if not t.exists():
        with write_txn(self.db):
            t.create(doc_id=str, store=str, page=int, version=int, route=str, text=str, flags=str, model=str, preset=str, at=float, blocks=str,
                     pk=('doc_id', 'store', 'page', 'version'))
    if 'blocks' not in t.columns_dict: self.db.execute('alter table read_pages add column blocks text')
    if 'stale' not in t.columns_dict: self.db.execute('alter table read_pages add column stale integer not null default 0')
    return t

@patch
def pages(self:Vault,
          doc_id:str,          # document whose pages to return
          history:bool=False,  # return every reading version
          stale:bool=False,    # include pages beyond the current file end
) -> L:
    "A read document's pages: the latest version of each, or every version with `history`."
    rs = L(self._pages_t().rows_where('doc_id=? and store=?', [doc_id, self.name], order_by='page, version'))
    if not stale: rs = rs.filter(lambda r: not r['stale'])
    if history: return rs.map(AttrDict)
    last = {}
    for r in rs: last[r['page']] = r
    return L(AttrDict(r, flags=json.loads(r['flags'] or '[]'), blocks=json.loads(r.get('blocks') or 'null')) for r in last.values())

@patch
def _file_pages(self:Vault, doc_id, rows, preset, count=None):
    t, now, have, new = self._pages_t(), time.time(), {r.page: r for r in self.pages(doc_id, stale=True)}, []
    with write_txn(self.db):
        if count is not None:
            self.db.execute('update read_pages set stale=(page >= ?) where doc_id=? and store=?', [count, doc_id, self.name])
        for r in rows:
            if (o := have.get(r.page)) and o.text == r.text and o.route == r.route and (o.blocks or None) == (r.get('blocks') or None): continue
            new.append(r.page)
            t.insert(dict(doc_id=doc_id, store=self.name, page=r.page, version=(o.version if o else 0) + 1, route=r.route, text=r.text,
                          flags=json.dumps(r.flags), model=r.get('model'), preset=preset if r.route == 'model' else None, at=now,
                          blocks=json.dumps(r['blocks'], ensure_ascii=False) if r.get('blocks') else None, stale=0))
    if new: self.scan_pii(doc_id, pages=new)

@patch
def prune_stale(self:Vault,
                doc_id:str=None,     # restrict to one document; None checks this shelf
                dry_run:bool=False,  # report pages without deleting their readings
) -> L:
    "Delete all versions of stale pages and return their document, page and version count."
    self._pages_t()
    where, args = 'store=? and stale=1', [self.name]
    if doc_id is not None: where += ' and doc_id=?'; args.append(doc_id)
    gone = L(self.db.q(f'select doc_id, page, count(*) as versions from read_pages where {where} group by doc_id, page order by doc_id, page', args))
    if not dry_run:
        with write_txn(self.db): self.db.execute(f'delete from read_pages where {where}', args)
    return gone

@patch
def _stale_view(self:Vault):
    "Copy this vault in memory and index retained stale pages there."
    view = Vault(':memory:', encoder=self.enc, store=self.name)
    try:
        with view.db.conn.backup('main', self.db.conn, 'main') as backup: backup.step()
        if view.db.t.usearch_indices.exists(): view.db.execute('update usearch_indices set path=null')
        if not view.db.t.read_pages.exists(): return view
        view._pages_t()
        targets = list(view.db.q('select distinct store, doc_id from read_pages where stale=1'))
        for r in targets:
            shelf = view if r['store'] == view.name else view.shelf(r['store'])
            d = shelf.doc(r['doc_id'])
            if d:
                marks = shelf.marks(d['id'])
                shelf.add([(p.page, p.text) for p in shelf.pages(d['id'], stale=True)], d['title'],
                          source=d['source'], kind=d['kind'], meta=d['meta'], force=True)
                shelf.t.docs.update(dict(id=d['id'], added_at=d['added_at']))
                if marks: shelf._marks().insert(marks, replace=True)
        return view
    except BaseException:
        view.db.close()
        raise

@patch
def _read_sent(self:Vault, doc_id, rows, model, chat, pii):
    from vishalakshi.ask import model_local
    sent = [AttrDict(doc_id=doc_id, page=r.page) for r in rows if r.route == 'model' and r.get('model') == getattr(chat, 'model', model)]
    self._sent(AttrDict(local=model_local(chat if chat is not None else model), send=sent, model=model or getattr(chat, 'model', None),
                        runtime=getattr(chat, 'runtime', None), policy=pii, kinds={}))

@patch
def read_doc(self:Vault, src, title:str=None, kind:str=None, parse:str='auto', model:str=None, preset:str='generic', script:str='', hint:str='',
         pages=None, dpi:int=200, workers:int=4, chat=None,
         pii:str='redact',      # off|local|redact|refuse, per page; the document's level and the shelf's floor apply
         allow:bool=False,      # allow once, for Personal pages
         local_model:str=None,  # reads the pages a hosted `model` may not
         local_chat=None,       # a built local reader instead of `local_model`
         progress=None,         # `progress(done, total, page, route, cost)` per file; False stops the reading
         spread:bool=False, strips:int=1, cache=None, redo=None, clean=None, tries:int=None,   # as `read_doc`; a folder's files cache in `cache/<path below the folder, extension included>`
         **kw):
    "Read a PDF, a page image or every one under a folder into the vault page by page, keeping each page's readings as versions."
    p, rk = Path(src), dict(spread=spread, strips=strips, redo=redo, clean=clean, tries=tries)
    if p.is_dir(): return doc_files(p).map(lambda f: self.read_doc(f, None, kind, parse, model, preset, script, hint, pages, dpi, workers, chat,
                                                                    pii=pii, allow=allow, local_model=local_model, local_chat=local_chat, progress=progress,
                                                                    cache=cache if cache is None or callable(cache) else Path(cache)/f.relative_to(p), **rk, **kw))
    lvl = self.level(str(p)).level if self.doc(str(p)) and self.doc(str(p))['source'] == str(p) else self.floor()
    rows = read_doc(p, parse, model, preset, title or '', script, hint, pages, dpi, self.assets(p.stem), workers, chat,
                    pii=pii, level=lvl, allow=allow, local_model=local_model, local_chat=local_chat, progress=progress, cache=cache, **rk)
    d0 = self.doc(str(p))
    have = {k: t for k, (t, _) in self._page_texts(d0['id']).items()} if d0 and Path(d0['source']).resolve() == p.resolve() else {}
    have = {k: t for k, t in have.items() if k in set(rows.attrgot('page'))}
    want = as_list(pages, int)
    ok = rows.filter(lambda r: (r.page not in {*rows.unread, *rows.failed} or r.page not in have) and not (r.get('blocked') and r.page in have)
                               and (want is None or r.page in want or r.page not in have))
    out = self.add(sorted({**have, **{r.page: r.text for r in ok}}.items()), title or p.stem, source=str(p), kind=kind or ('pdf' if p.suffix.lower() == '.pdf' else 'image'),
                   meta=dict(parse=parse, model=model, preset=preset, script=script, hint=hint), force=True, **kw)
    self._file_pages(out['doc_id'], ok, preset, len(rows))
    self._read_sent(out['doc_id'], rows, model, chat, pii)
    blocked = [r.page for r in rows if r.get('blocked')]
    return dict(out, routes={k: len(v) for k, v in rows.groupby(lambda r: r.route).items()},
                flagged=[r.page for r in rows if r.flags], blocked=blocked,
                kind=first(r.blocked.kind for r in rows if r.get('blocked')), cost=doc_cost(rows),
                model_pages=len(rows.filter(lambda r: r.route == 'model')), cancelled=rows.cancelled, unread=rows.unread, failed=rows.failed)

@patch
def reparse(self:Vault, doc_id:str, pages=None, model:str=None, preset:str=None, script:str=None, hint:str=None, parse:str='llm',
            dpi:int=200, workers:int=4, chat=None, pii:str='redact', allow:bool=False, local_model:str=None, local_chat=None,
            progress=None) -> dict:
    "Read pages of a document again, all of them when `pages` is None, file the new readings as new versions and re-index the document."
    d, pages = first(self.t.docs(where='id=?', where_args=[doc_id])), as_list(pages, int)
    assert d, f'no document {doc_id}'
    m = json.loads(d['meta'] or '{}')
    preset, script, hint = ifnone(preset, m.get('preset', 'generic')), ifnone(script, m.get('script', '')), ifnone(hint, m.get('hint', ''))
    old = {r.page: r for r in self.pages(doc_id)}
    got = read_doc(d['source'], parse, model, preset, d['title'], script, hint, pages, dpi, self.assets(Path(d['source']).stem), workers, chat,
                   pii=pii, level=self.level(doc_id).level, allow=allow, local_model=local_model, local_chat=local_chat, progress=progress)
    keep = (got if pages is None else got.filter(lambda r: r.page in set(pages))).filter(lambda r: r.page not in got.unread)
    blocked = keep.filter(lambda r: r.get('blocked'))
    keep = keep.filter(lambda r: not r.get('blocked'))
    self._file_pages(doc_id, keep, preset, len(got))
    self._read_sent(doc_id, keep, model, chat, pii)
    new = {r.page: r for r in self.pages(doc_id)}
    out = self.add([(k, new[k].text) for k in sorted(new)], d['title'], source=d['source'], kind=d['kind'], meta=dict(m, reparsed=time.time()), force=True)
    return dict(out, pages=sorted(r.page for r in keep), changed=[k for k in new if old.get(k) is None or old[k].text != new[k].text],
                blocked=[r.page for r in blocked], kind=first(r.blocked.kind for r in blocked), cost=doc_cost(keep),
                model_pages=len(keep.filter(lambda r: r.route == 'model')), cancelled=got.cancelled, unread=got.unread)

In [ ]:
_v = Vault(Path(tempfile.mkdtemp())/'v.db', offline=True)
_o = _v.read_doc(_scan, parse='text')
test_eq(_o['routes'], {'scanned': 1}); test_eq(_o['flagged'], [0])
test_eq([(p.page, p.version, p.route) for p in _v.pages(_o['doc_id'])], [(0, 1, 'scanned')])
_r = _v.reparse(_o['doc_id'], chat=Fake())
test_eq(_r['changed'], [0]); test_eq([p.version for p in _v.pages(_o['doc_id'], history=True)], [1, 2])
test_eq(_v.pages(_o['doc_id'])[0].route, 'model')
test_eq([b['kind'] for b in _v.pages(_o['doc_id'])[0].blocks], ['heading', 'prose'])
assert 'dominant sequence' in _v.search('dominant sequence transduction')[0]['snippet']
_v.read_doc(_scan, parse='text'); test_eq(len(_v.pages(_o['doc_id'], history=True)), 3)   # read again as text: the text route is a new version
test_eq(len(_v.read_doc(_d, parse='text')), 2)

In [ ]:
# the vault's results carry the cost, the model pages and a cancel; a cancelled reparse leaves its unread pages alone
_cv = Vault(Path(tempfile.mkdtemp())/'c.db', offline=True)
_o = _cv.read_doc(_src, parse='llm', chat=Fake(), workers=1)
test_eq((_o['cost'], _o['model_pages'], _o['cancelled'], _o['unread'], _o['failed']), (.02, 2, False, [], []))
Fake.calls, _vc = 0, Path(tempfile.mkdtemp())/'read'
_cv.read_doc(_src, parse='llm', chat=Fake(), workers=1, cache=_vc); _o = _cv.read_doc(_src, parse='llm', chat=Fake(), workers=1, cache=_vc)
test_eq((Fake.calls, _o['cost'], _o['model_pages']), (2, 0, 2))   # the second read is the cache's
test_eq([p.version for p in _cv.pages(_o['doc_id'])], [1, 1])       # and files no new version: its pages are as they were
_o = _cv.read_doc(_src, parse='llm', chat=Fake(), workers=1, progress=lambda d, *a: d < 1)
test_eq((_o['cost'], _o['model_pages'], _o['cancelled'], _o['unread'], _o['routes']), (.01, 1, True, [1], {'model': 1, 'text': 1}))
_before = _cv.pages(_o['doc_id'], history=True)
_r = _cv.reparse(_o['doc_id'], chat=Fake(), workers=1, progress=lambda d, *a: d < 1)
test_eq((_r['pages'], _r['cost'], _r['model_pages'], _r['cancelled'], _r['unread']), ([0], .01, 1, True, [1]))
test_eq(_cv.pages(_o['doc_id'])[1].route, 'model')   # the unread page keeps its earlier reading
test_eq(_cv.document(_o['doc_id']).text.split(), ' '.join(p.text for p in _cv.pages(_o['doc_id'])).split())   # and the index agrees
_r = _cv.reparse(_o['doc_id'], chat=Boxed(), pages=[1])
test_eq((_r['cost'], _r['cancelled']), (0, False)); test_eq(_cv.pages(_o['doc_id'])[1].blocks[0]['box'], [.1, .05, .9, .1])
_s = _cv.read_doc(_d, parse='text'); test_eq(L(_s).map(lambda o: (o['cost'], o['model_pages'])), [(0, 0), (0, 0)])
_cf, _cc = Path(tempfile.mkdtemp()), Path(tempfile.mkdtemp())
for _n in ('a', 'b'): (_cf/_n).mkdir(); (_cf/_n/'pg.png').write_bytes(_png)
_cv.read_doc(_cf, parse='llm', chat=Fake(), workers=1, cache=_cc)
test_eq(sorted(str(f.relative_to(_cc)) for f in _cc.rglob('p0001.json')), ['a/pg.png/p0001.json', 'b/pg.png/p0001.json'])   # same stem, own cache
_pv = Vault(Path(tempfile.mkdtemp())/'p.db', offline=True)
_o = _pv.read_doc(_src, parse='llm', chat=Fake(), workers=1)
_was = {p.page: (p.version, p.route, p.text) for p in _pv.pages(_o['doc_id'])}
_pv.read_doc(_src, parse='llm', chat=Fake(), workers=1, pages=[1], clean=str.upper)
_now = {p.page: (p.version, p.route, p.text) for p in _pv.pages(_o['doc_id'])}
test_eq(_now[0], _was[0]); assert _now[1][0] == 2   # a page that was not asked for keeps its stored reading; the asked one is a new version

In [ ]:
#| hide
# a page goes to a hosted reader only as its level allows: an image cannot be masked
class Local(Fake):
    "A reader on this machine."
    runtime, model = 'litert', 'litert/vision'
class Hosted(Fake):
    "A hosted reader that names its runtime."
    runtime, model = 'remote', 'remote/vision'
Fake.calls = 0
_b = read_page(_png, chat=Hosted(), text='write to ada@example.com')
test_eq((_b.blocked.kind, _b.text, Fake.calls), ('pii_blocked', '', 0))
test_eq(read_page(_png, chat=Hosted(), text='write to ada@example.com', allow=True).text.startswith('## Attention'), True)
# the title goes into the reading prompt, so it is judged with the page: a scan with no text is not let through on that
Fake.calls = 0
test_eq((read_page(_png, chat=Hosted(), title='Payslip for ada@example.com').blocked.kind, Fake.calls), ('pii_blocked', 0))
test_eq(read_page(_png, chat=Local(), title='Payslip for ada@example.com').get('blocked'), None)
test_eq(read_page(_png, chat=Hosted(), level='local', allow=True).blocked.kind, 'local_only')
test_eq(read_page(_png, chat=Local(), level='local').get('blocked'), None)
test_eq(read_page(_png, chat=Hosted(), text='write to ada@example.com', pii='off').get('blocked'), None)   # off sends a Personal page

# a scanned page in a Local only shelf never reaches the hosted reader: the local one reads it, or nothing does
_lv = Vault(Path(tempfile.mkdtemp())/'l.db', offline=True)
_lv.set_floor('local')
Fake.calls = 0
_o = _lv.read_doc(_scan, parse='llm', chat=Hosted())
test_eq((_o['blocked'], _o['kind'], Fake.calls), ([0], 'local_only', 0))
_o = _lv.reparse(_o['doc_id'], chat=Hosted(), local_chat=Local())
test_eq((_o['blocked'], _o['changed'], _lv.pages(_o['doc_id'])[0].model), ([], [0], 'litert/vision'))
test_eq(_lv.sends(_o['doc_id']), [])                                    # nothing left the machine
# a model reading kept from before survives a reread that blocks its page: the scan's empty text layer must not replace it
_was = _lv.pages(_o['doc_id'])[0]
assert _was.text
_o = _lv.read_doc(_scan, parse='llm', chat=Hosted())
test_eq(_o['blocked'], [0])
_now = _lv.pages(_o['doc_id'])[0]
test_eq((_now.version, _now.model, _now.text), (_was.version, 'litert/vision', _was.text))
# a text layer that reads as a passport makes the document Local only before any page goes anywhere
_pp = _d/'passport.pdf'
Pdf.from_markdown('PASSPORT\n\nType P  Country code GBR  Passport No. 123456789\n\nSurname SMITH\n\nGiven names JOHN\n\n'
                  'Nationality BRITISH CITIZEN\n\nDate of birth 01 JAN 1980\n\nDate of expiry 05 MAR 2029').save(str(_pp))
test_eq(read_doc(_pp, parse='llm', chat=Hosted(), out=_d/'.out')[0].blocked.kind, 'local_only')
# a model's reading is scanned and its findings filed against the page's new version
class Says(Local):
    def __call__(self, msg): return dict(content='{"items":[{"kind":"prose","text":"Write to ada@example.com about it."}]}')
_ov = Vault(Path(tempfile.mkdtemp())/'o.db', offline=True)
_o = _ov.read_doc(_scan, parse='llm', chat=Says())
test_eq([(f.kind, f.version) for f in _ov.findings(_o['doc_id'])], [('email', 1)])
test_eq(_ov.level(_o['doc_id'], page=0).level, 'personal')
# a hosted read is logged, page by page
_hv = Vault(Path(tempfile.mkdtemp())/'h.db', offline=True)
_o = _hv.read_doc(_scan, parse='llm', chat=Hosted())
test_eq((_hv.sends(_o['doc_id'])[0].pages, _hv.sends(_o['doc_id'])[0].runtime), ([0], 'remote'))


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

A three-page PDF shortened to two pages keeps its last page as stale. Both read paths exclude that page from retrieval. A dry run preserves every reading. Pruning removes them.

In [ ]:
with tempfile.TemporaryDirectory() as _short_dir:
    _short_root = Path(_short_dir)
    _short_files = []
    for _page_no, _text in enumerate(['Orchard apples grow in the sunny garden.', 'Sailing boats cross the blue ocean.', 'Zebramarker belongs to the vanished last page.']):
        _page_path = _short_root/f'p{_page_no}.pdf'
        Pdf.from_markdown(_text).save(str(_page_path))
        _short_files.append(str(_page_path))
    _short_src = _short_root/'book.pdf'
    _short_vault = Vault(_short_root/'vault.db', offline=True)
    for _read_again in ('read_doc', 'reparse'):
        Pdf.merge(_short_files).save(str(_short_src))
        _short_id = _short_vault.read_doc(_short_src, parse='text')['doc_id']
        Pdf.merge(_short_files[:2]).save(str(_short_src))
        if _read_again == 'read_doc': _short_vault.read_doc(_short_src, parse='text')
        else: _short_vault.reparse(_short_id, parse='text')
        test_eq(_short_vault.pages(_short_id).attrgot('page'), [0, 1])
        _retained = _short_vault.pages(_short_id, stale=True)
        test_eq(_retained.attrgot('page'), [0, 1, 2])
        assert _retained[2].stale
        assert 'Zebramarker' not in _short_vault.document(_short_id).text
        assert 'Zebramarker' in _short_vault.document(_short_id, stale=True).text
        test_eq(_short_vault.search('Zebramarker', where="content like '%Zebramarker%'"), [])
        assert any('Zebramarker' in h.snippet for h in _short_vault.search('Zebramarker', stale=True))
        assert 'Zebramarker' not in str(_short_vault.sections('Zebramarker'))
        assert 'Zebramarker' in str(_short_vault.sections('Zebramarker', stale=True))
        assert 'Zebramarker' not in str(_short_vault.context('Zebramarker', code=0, shelves=0).results)
        assert 'Zebramarker' in str(_short_vault.context('Zebramarker', code=0, shelves=0, stale=True).results)
        assert 'Zebramarker' not in _short_vault.doc_context(_short_id, 'what remains?', related=0).doc.text
        assert 'Zebramarker' in _short_vault.doc_context(_short_id, 'what remains?', related=0, stale=True).doc.text
        assert 'Zebramarker' not in str(_short_vault.federate('Zebramarker', repo=False, env=False, grep=False, shelves=None).hits)
        assert 'Zebramarker' in str(_short_vault.federate('Zebramarker', repo=False, env=False, grep=False, shelves=None, stale=True).hits)
        assert 'Zebramarker' not in _short_vault.document(_short_id).text
        test_eq(_short_vault.search('Zebramarker', where="content like '%Zebramarker%'"), [])
        _preview = _short_vault.prune_stale(_short_id, dry_run=True)
        test_eq([r['page'] for r in _preview], [2])
        test_eq(len(_short_vault.pages(_short_id, stale=True)), 3)
        test_eq(_short_vault.prune_stale(), _preview)
        test_eq(_short_vault.pages(_short_id, history=True, stale=True).attrgot('page'), [0, 1])
    _paper_vault = _short_vault.shelf('papers')
    Pdf.merge(_short_files).save(str(_short_src))
    _paper_id = _paper_vault.read_doc(_short_src, parse='text')['doc_id']
    Pdf.merge(_short_files[:2]).save(str(_short_src))
    _paper_vault.reparse(_paper_id, parse='text', pages=[0])
    _paper_vault.mark_stale(_paper_id, reason='changed source')
    _paper_age = _paper_vault.doc(_paper_id)['added_at']
    _sidecars = {p: p.read_bytes() for p in _short_root.glob('*.usearch*')}
    assert 'Zebramarker' in _paper_vault.document(_paper_id, stale=True).text
    assert any(h.stale for h in _paper_vault.search('Zebramarker', stale=True))
    test_eq(_paper_vault.doc(_paper_id)['added_at'], _paper_age)
    assert all(p.read_bytes() == data for p, data in _sidecars.items())
    test_eq(_paper_vault.pages(_paper_id).attrgot('page'), [0, 1])
    _short_vault.db.close()
_preview